# M5 — The table map (Neo4j knowledge graph) + version D

**Goal of this step:** fix version B's biggest weakness. Search finds the *main* table, but misses the
*connecting* tables needed for a join.

**Analogy:** search is like asking *"which station is closest to my destination?"* It gives you the right station,
but not the route. The **table map** is the subway map: it knows which tables connect to which (the lines), which
tables are look-alikes (stations with almost the same name), and what everyday words mean ("heavy shoppers" →
the user table).

**A knowledge graph** stores **things** (nodes) and **how they're connected** (relationships).
**Neo4j** is a graph database, and **Cypher** is its query language. It reads like drawing arrows:
`(a:Table)-[:JOINS_WITH]->(b:Table)`.

**What's in our map**
| Things (nodes) | Connections (relationships) |
|---|---|
| `Table` (28): grain, freshness, traffic | `(Table)-[:JOINS_WITH {on}]->(Table)`: can be joined on this column |
| `Column` (166) | `(Table)-[:LOOKALIKE_OF {differs_by}]->(Table)`: similar, but a different answer |
| `Metric` (6): impressions, clicks, CTR, reach … | `(Table)-[:ROLLS_UP_TO]->(Table)`: hourly → daily → weekly |
| `Entity` (8): advertiser, campaign, ad, brand … | `(Entity)-[:HOME_TABLE]->(Table)`, `(advertiser)-[:OWNS]->(campaign)-[:OWNS]->(ad)` |
| `Term` (57): words people use | `(Term)-[:MEANS]->(Metric or Entity)`: "heavy shoppers" → user |

**Version D** = B's 5 search hits **+** what the map adds (lookup tables for business words, the tables on the join
route, look-alikes), at most 8 cards.

**Files in this step**
| File | What it does |
|---|---|
| `catalog/load_neo4j.py` | builds the map from the table cards |
| `catalog/graph.py` | `get_join_path`, `list_lookalikes`, `tables_for_terms`, `expand_tables` (D's rule) |
| `catalog/check_graph.py` | the done-test: 10 join-route checks |
| `agent/pipelines.py` | adds `version_d()` |
| `configs/D.yaml` | D's settings |

**Done when:** `get_join_path` finds the right bridge tables on 10 checks ✅

**See the map yourself:** open http://localhost:7474 in your browser. User `neo4j`, password = `NEO4J_PASSWORD`
in your `.env` file. Then type `MATCH (n) RETURN n` and press ▶.

In [1]:
import sys, json, glob
for folder in ["../agent", "../catalog", "../harness", "../mcp_server"]:
    sys.path.append(folder)

## 1. Build the map
Neo4j must be running. If it isn't, start it in a terminal with `/opt/homebrew/opt/neo4j/bin/neo4j start`
(and stop it with `.../neo4j stop`).

In [2]:
from load_neo4j import build_graph
build_graph()

Nodes:         {'Column': 166, 'Entity': 8, 'Metric': 6, 'Table': 28, 'Term': 57}
Relationships: {'HAS_COLUMN': 166, 'HAS_METRIC': 68, 'HOME_TABLE': 8, 'JOINS_WITH': 55, 'LOOKALIKE_OF': 42, 'MEANS': 57, 'OWNS': 2, 'ROLLS_UP_TO': 5}


## 2. Ask the map questions with Cypher
`query()` sends a Cypher query to Neo4j and returns the rows.

**Which tables does `agg_campaign_daily` join to, and on which column?**

In [3]:
from graph import query
for r in query("""
    MATCH (:Table {name: 'agg_campaign_daily'})-[j:JOINS_WITH]-(other:Table)
    RETURN other.name AS table, j.on AS on
"""):
    print(f"{r['table']:20s} on {r['on']}")

/Users/anshikapuneet/Documents/Claude_history/capstone_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dim_advertiser       on advertiser_id
dim_campaign         on campaign_id
dim_date             on event_date = date


**What are its look-alikes, and how does each one differ?**

In [4]:
from graph import list_lookalikes
for table, differs in list_lookalikes("agg_campaign_daily"):
    print(f"{table:28s} differs by: {', '.join(differs)}")

reach_campaign_daily         differs by: what is counted
agg_campaign_daily_valid     differs by: traffic
agg_campaign_weekly          differs by: grain
agg_campaign_hourly          differs by: grain, freshness


**What do everyday words point to?** Business word → entity → its home table:

In [5]:
for r in query("""
    MATCH (w:Term)-[:MEANS]->(e:Entity)-[:HOME_TABLE]->(t:Table)
    WHERE w.text IN ['heavy shoppers', 'shoppers', 'gender', 'category', 'weekend', 'advertiser']
    RETURN w.text AS word, e.name AS entity, t.name AS table
"""):
    print(f"{r['word']:10s} -> {r['entity']:10s} -> {r['table']}")

advertiser -> advertiser -> dim_advertiser
category   -> category   -> dim_ad
shoppers   -> user       -> dim_user
gender     -> user       -> dim_user
weekend    -> date       -> dim_date


## 3. The key trick: the join route
`get_join_path(a, b)` asks Neo4j for the **shortest route** between two tables (at most 3 steps).
The tables *in the middle* are the bridges a query needs.

In [6]:
from graph import get_join_path
print(" -> ".join(get_join_path("dim_user", "agg_campaign_daily_by_audience")))
print(" -> ".join(get_join_path("agg_brand_daily", "agg_category_daily")))

dim_user -> fact_ad_events -> dim_campaign -> agg_campaign_daily_by_audience
agg_brand_daily -> dim_ad -> agg_category_daily


The done-test: 10 routes that must go through the right bridge table.

In [7]:
from check_graph import run_checks
run_checks()

OK   dim_user -> agg_campaign_daily
      route: dim_user -> fact_ad_events -> dim_advertiser -> agg_campaign_daily
OK   agg_brand_daily -> agg_category_daily
      route: agg_brand_daily -> dim_ad -> agg_category_daily
OK   agg_campaign_daily -> dim_date
      route: agg_campaign_daily -> dim_date
OK   agg_ad_daily -> dim_advertiser
      route: agg_ad_daily -> dim_campaign -> dim_advertiser
OK   ref_suspicious_user_hours -> fact_ad_events
      route: ref_suspicious_user_hours -> fact_ad_events
OK   dim_user -> dim_ad
      route: dim_user -> fact_ad_events -> dim_ad
OK   reach_campaign_total -> dim_advertiser
      route: reach_campaign_total -> dim_campaign -> dim_advertiser
OK   live_campaign_last6h -> dim_campaign
      route: live_campaign_last6h -> dim_campaign
OK   agg_campaign_daily_by_audience -> dim_user
      route: agg_campaign_daily_by_audience -> dim_date -> fact_ad_events -> dim_user
OK   dim_user -> dim_placement
      route: dim_user -> fact_ad_events -> dim_placemen

## 4. What B sees vs what D sees
The same question as in M4, where B failed because it never saw the user table:

In [8]:
from search import search_tables
from graph import tables_for_terms, expand_tables

q = "What was the CTR among heavy shoppers (shopping depth 3) for campaign 12519 yesterday?"
print(q)
print("B sees  :", [t for t, _ in search_tables(q, k=5)])
print("words   :", tables_for_terms(q))
print("D sees  :", expand_tables(q))

What was the CTR among heavy shoppers (shopping depth 3) for campaign 12519 yesterday?


B sees  : ['agg_campaign_daily_valid', 'agg_campaign_daily_by_audience', 'agg_campaign_hourly_valid', 'agg_campaign_daily', 'agg_ad_daily_valid']
words   : ['dim_campaign', 'dim_user']
D sees  : ['agg_campaign_daily_valid', 'agg_campaign_daily_by_audience', 'agg_campaign_hourly_valid', 'agg_campaign_daily', 'agg_ad_daily_valid', 'dim_campaign', 'dim_user', 'dim_advertiser']


The map spotted "shoppers" (→ `dim_user`) and "campaign" (→ `dim_campaign`), so D now sees the user table that B never saw. **But look what's missing:** `fact_ad_events`, the bridge that links users to campaigns. The 5 search hits + 2 word tables + 1 bridge already fill the **8-card limit**, so the most important bridge is cut off. That's why this question is still wrong below.

This is a real trade-off: more search hits help easy questions, but leave less room for bridges. Better search (version C, M8) or the agent asking for a join route itself (E, M7) are the planned fixes.

## 5. Results on the practice questions: A vs B vs D
```
.venv/bin/python harness/run.py configs/D.yaml --split practice
```

In [9]:
def latest(version):
    folder = sorted(glob.glob(f"../results/*_{version}_practice"))[-1]
    return folder, json.load(open(folder + "/metrics.json"))

runs = {v: latest(v) for v in ["A", "B", "D"]}
keys = ["accuracy", "accuracy_easy", "accuracy_medium", "accuracy_tricky", "table_f1",
        "gold_tables_shown", "avg_tokens_in", "avg_seconds", "avg_cost_usd"]
print(f"{'metric':20s}" + "".join(f"{v:>10s}" for v in runs))
for k in keys:
    print(f"{k:20s}" + "".join(f"{runs[v][1][k]:>10}" for v in runs))

metric                       A         B         D
accuracy                 0.875       0.7      0.85
accuracy_easy              1.0       1.0       1.0
accuracy_medium         0.8462    0.3077    0.6923
accuracy_tricky         0.7857    0.7857    0.8571
table_f1                0.9542    0.7208    0.8292
gold_tables_shown          1.0      0.65      0.85
avg_tokens_in           3208.8   712.825   911.225
avg_seconds             3.7472    4.2538    3.7412
avg_cost_usd            0.0012    0.0007    0.0007


## 6. D's remaining mistakes

In [10]:
for line in open(runs["D"][0] + "/answers.jsonl"):
    r = json.loads(line)
    if not r["correct"]:
        print(f"{r['id']} {r['template']:3s} needed tables shown: {str(r['gold_tables_shown']):5s}  {r['question'][:70]}")
        print(f"      used: {r['tables_used']}   gold: {r['gold_tables']}")

q047 M2  needed tables shown: False  What was the CTR among heavy shoppers (shopping depth 3) for campaign 
      used: ['agg_campaign_daily_by_audience']   gold: ['fact_ad_events', 'dim_user']
q057 M5  needed tables shown: False  Which brand in category 4357 had the most clicks yesterday?
      used: ['agg_brand_daily', 'dim_ad']   gold: ['agg_ad_daily', 'dim_ad']
q058 M5  needed tables shown: True   Top brand by clicks yesterday within category 6208?
      used: ['agg_brand_daily', 'dim_ad']   gold: ['agg_ad_daily', 'dim_ad']
q060 M5  needed tables shown: True   Top brand by clicks yesterday within category 6142?
      used: ['agg_brand_daily', 'dim_ad']   gold: ['agg_ad_daily', 'dim_ad']
q101 T6  needed tables shown: True   How many clicks did campaign 319897 get in the week starting Monday 8 
      used: ['agg_campaign_weekly']   gold: ['agg_campaign_weekly']
q102 T6  needed tables shown: True   Weekly clicks for campaign 133088, week of 8 May?
      used: ['agg_campaign_weekly']  

## 7. What this tells us
- **The map works for joins.** D gets about twice as many join (medium) questions right as B, and it shows the
  needed tables much more often. That's the GraphRAG idea in the design doc, working.
- **D is cheap.** It reads about 900 tokens of cards versus A's 3,200, with similar speed and cost.
- **A is still ahead at 28 tables** (87.5% vs 85%, a single question's difference on 40 questions, so not a real
  difference yet). The bigger-catalog test (M8) shows what happens at 100–250 tables, where A has to read
  everything.
- **The 8-card limit cuts off bridges.** In q047, the bridge `fact_ad_events` didn't fit (section 4).
- **Most other mistakes are about *meaning*, not missing tables.** "Top brand within a category" needs
  per-ad clicks for that category, but the AI used the brand's clicks across *all* categories (`agg_brand_daily`).
  More cards can't fix that. The agent (E, M7) can check sample rows and retry.

**One rule change, made on practice questions:** the first D run started from only 3 search hits and scored 77.5%,
because it lost tables that B had found. D was changed to start from B's 5 hits (so it only *adds* to B), which
scored 85%. Practice questions are meant for this kind of fix, and it's noted in the README. The test questions
haven't been used.

**Next step (M6):** put all the tools (search, join route, look-alikes, describe table, sample rows, run SQL)
behind one MCP server.